# OPTIMIZED PIPELINE 1: MICE + UNDERSAMPLING (BASELINE)

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import RobustScaler
from imblearn.under_sampling import RandomUnderSampler

DATA_DIR = r"c:\Users\edgib\Downloads\PREDICTIVE MODEL USING NEURAL NETWORK"
INPUT_FILE = os.path.join(DATA_DIR, "accepted_2007_to_2018Q4.csv")
OUT_DIR = r"c:\Users\edgib\Downloads\PREDICTIVE MODEL USING NEURAL NETWORK\OPTIMIZED_CLEANING_PIPELINES"
RANDOM_SEED = 42

In [ ]:
# 1. LOAD RAW DATA
df = pd.read_csv(INPUT_FILE, low_memory=False)

# 2. FILTER RESOLVED LOANS & CREATE TARGET
resolved_statuses = ["Fully Paid", "Charged Off"]
df = df[df["loan_status"].isin(resolved_statuses)].copy()
df["target"] = (df["loan_status"] == "Charged Off").astype(int)
df.drop(columns=["loan_status"], inplace=True)

In [ ]:
# 3-5. DROP NULL, CONSTANT, AND LEAKAGE COLUMNS
missing_cols = df.columns[df.isnull().mean() == 1].tolist()
df.drop(columns=missing_cols, inplace=True)

constant_cols = [col for col in df.columns if df[col].nunique() == 1]
df.drop(columns=constant_cols, inplace=True)

hardship_cols = [col for col in df.columns if "hardship" in col.lower() or "settlement" in col.lower()]
df.drop(columns=hardship_cols, inplace=True)

time_after_loan_cols = ["total_pymnt","total_pymnt_inv","total_rec_prncp","total_rec_int", "total_rec_late_fee", "recoveries",
    "collection_recovery_fee", "last_pymnt_d", "last_pymnt_amnt",
    "last_credit_pull_d", "last_fico_range_high", "last_fico_range_low"]
df.drop(columns=time_after_loan_cols, inplace=True, errors='ignore')

imm_after_app_cols = ["funded_amnt", "funded_amnt_inv", "url", "initial_list_status", "disbursement_method"]
df.drop(columns=imm_after_app_cols, inplace=True, errors='ignore')

In [ ]:
# 6-7. DROP ID COLUMNS AND >40% MISSING COLUMNS
id_text_cols = ["id","emp_title","title","zip_code","desc"]
df.drop(columns=id_text_cols, inplace=True, errors='ignore')

high_missing = df.columns[df.isnull().mean() > 0.4].tolist()
df.drop(columns=high_missing, inplace=True)

In [ ]:
# 8. CATEGORICAL ENCODING
df["term"] = df["term"].astype(str).str.strip().str.replace("months","").astype(int)
df.dropna(subset=['emp_length'], inplace=True)
emp_map = {"< 1 year": 0, "1 year": 1, "2 years": 2, "3 years": 3, "4 years": 4, "5 years": 5,
           "6 years": 6, "7 years": 7, "8 years": 8, "9 years": 9, "10+ years": 10}
df['emp_length'] = df['emp_length'].map(emp_map)

df.drop(columns=['grade'], inplace=True, errors='ignore')
sub_grade_map = {f"{l}{n}": i+1 for i, (l, n) in enumerate([(l, n) for l in "ABCDEFG" for n in range(1,6)])}
df['sub_grade'] = df['sub_grade'].map(sub_grade_map)

for col in ['home_ownership', 'verification_status', 'purpose']:
    df = pd.get_dummies(df, columns=[col], drop_first=True, dtype=int)

df['earliest_cr_line'] = pd.to_datetime(df['earliest_cr_line'])
df['issue_d'] = pd.to_datetime(df['issue_d'])
df['mnths_since_earliest_cr'] = (df['issue_d'].dt.year - df['earliest_cr_line'].dt.year)*12 + (df['issue_d'].dt.month - df['earliest_cr_line'].dt.month)
df.drop(columns=['earliest_cr_line','issue_d'], inplace=True)
df['application_type'] = df['application_type'].map({'Individual':0,'Joint App':1})

In [ ]:
# 9-10. ROW FILTERS
cols_less_1 = df.columns[(df.isnull().mean() > 0) & (df.isnull().mean() <= 0.01)]
df.dropna(subset=cols_less_1, inplace=True)
df = df[df.isnull().sum(axis=1) <= 6].copy()

In [ ]:
# 11. TRAIN / TEST SPLIT
X = df.drop(columns=['target'])
y = df['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=RANDOM_SEED, stratify=y)

In [ ]:
# 12-14. MICE IMPUTATION AND CLIPPING
num_cols = X_train.columns.drop('addr_state')
mice = IterativeImputer(max_iter=10, random_state=RANDOM_SEED)

X_train[num_cols] = mice.fit_transform(X_train[num_cols])
X_test[num_cols] = mice.transform(X_test[num_cols])

for col in num_cols:
    X_train[col] = X_train[col].clip(lower=0)
    X_test[col] = X_test[col].clip(lower=0)

for col in ['percent_bc_gt_75','pct_tl_nvr_dlq']:
    X_train[col] = X_train[col].clip(upper=100.0)
    X_test[col] = X_test[col].clip(upper=100.0)
X_train['bc_util'] = X_train['bc_util'].clip(upper=340.0)
X_test['bc_util'] = X_test['bc_util'].clip(upper=340.0)
X_train['mths_since_recent_inq'] = X_train['mths_since_recent_inq'].clip(upper=25.0)
X_test['mths_since_recent_inq'] = X_test['mths_since_recent_inq'].clip(upper=25.0)

In [ ]:
# *** EXPORT POST-MICE DATASETS (CSV) FOR PIPELINE 02 ***
X_train.to_csv(os.path.join(OUT_DIR, 'X_TRAIN_POST_MICE.csv'), index=False)
X_test.to_csv(os.path.join(OUT_DIR, 'X_TEST_POST_MICE.csv'), index=False)
y_train.to_frame().to_csv(os.path.join(OUT_DIR, 'Y_TRAIN_RAW.csv'), index=False)
y_test.to_frame().to_csv(os.path.join(OUT_DIR, 'Y_TEST_RAW.csv'), index=False)
print("Intermediate post-MICE datasets saved (CSV) for Pipeline 2.")

In [ ]:
# 15. UNDERSAMPLING (FOR THIS PIPELINE ONLY)
rus = RandomUnderSampler(sampling_strategy=1.0, random_state=RANDOM_SEED)
X_train_bal, y_train_bal = rus.fit_resample(X_train, y_train)

In [ ]:
# 16-17. TARGET ENCODING AND SCALING
state_means = y_train_bal.groupby(X_train_bal['addr_state']).mean()
X_train_bal['addr_state'] = X_train_bal['addr_state'].map(state_means)
X_test['addr_state'] = X_test['addr_state'].map(state_means)

scaler = RobustScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train_bal), columns=X_train_bal.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

In [ ]:
# 18. CORRELATION FILTER
train_full = X_train_scaled.copy()
train_full['TARGET'] = y_train_bal.values

corr_mat = X_train_scaled.corr().abs()
corr_target = train_full.corr().abs()['TARGET']
upper_tri = corr_mat.where(np.triu(np.ones(corr_mat.shape), k=1).astype(bool))

cols_to_drop = set()
for col in upper_tri.columns:
    for corr_col in upper_tri.index[upper_tri[col] > 0.9]:
        if corr_target[col] < corr_target[corr_col]:
            cols_to_drop.add(col)
        else:
            cols_to_drop.add(corr_col)

X_train_final = X_train_scaled.drop(columns=list(cols_to_drop) + ['purpose_educational'], errors='ignore')
X_test_final = X_test_scaled.drop(columns=list(cols_to_drop) + ['purpose_educational'], errors='ignore')

In [ ]:
# 19. EXPORT FINAL DATASETS (BASELINE)
X_train_final.to_parquet(os.path.join(OUT_DIR, 'X_TRAIN_BASELINE.parquet'), index=False)
X_test_final.to_parquet(os.path.join(OUT_DIR, 'X_TEST_BASELINE.parquet'), index=False)
y_train_bal.to_frame().to_parquet(os.path.join(OUT_DIR, 'Y_TRAIN_BASELINE.parquet'), index=False)
y_test.to_frame().to_parquet(os.path.join(OUT_DIR, 'Y_TEST_BASELINE.parquet'), index=False)
print("Baseline (Undersampled) Pipeline Complete.")